# Set-up  

In [1]:
!pip install braintrust autoevals

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.1/359.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive, userdata
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google import genai
from google.genai import types
import time

import pandas as pd

# Gemini Client Initiation

In [4]:
client = genai.Client(api_key=userdata.get('gemini_api_key'))
client

In [8]:
if len(client.file_search_stores.list()) < 2:
  # File name will be visible in citations
  file_search_store = client.file_search_stores.create(
      config={'display_name': 'Legal Information'
      })
  FILE_STORE_NAME = file_search_store.name
FILE_STORE_NAME

'fileSearchStores/legal-information-iha7u4ocs45v'

In [9]:
# List all file search stores
for fs in client.file_search_stores.list():
    print(f" {fs.name} - {fs.display_name}")

 fileSearchStores/legal-information-jjbq4kr8yz7z - Legal Information
 fileSearchStores/legal-information-iha7u4ocs45v - Legal Information


In [10]:
# FILE_STORE_NAME = "fileSearchStores/legal-information-jjbq4kr8yz7z"

FILE_STORE_NAME = "fileSearchStores/legal-information-iha7u4ocs45v" ## New One

In [ ]:
# client.file_search_stores.delete(name='fileSearchStores/botpolicy20240702-w2uusfol5kgo', config={'force':True})

# Add File To File Search Store

In [11]:
def store_file(cilent, file_name, file_search_store_name, file_path='/content/drive/MyDrive/KKP/'):
  full_file_path = f"{file_path}{file_name}"
  print(f"Storing File: {full_file_path}")


  operation = client.file_search_stores.upload_to_file_search_store(
    file=full_file_path,
    file_search_store_name=file_search_store_name,
    config={
        'display_name' : file_name,
    }
  )
  return operation

In [12]:
file_list = ['The Civil and Commercial Code_20251111.pdf'
  # '20240702_BOT_Policy.pdf',
  # 'Analysis_of_Legal1.pdf'
  ]

for file_name in file_list:
  operation = store_file(client, file_name, FILE_STORE_NAME)

  while not operation.done:
      time.sleep(5)
      operation = client.operations.get(operation)


Storing File: /content/drive/MyDrive/KKP/The Civil and Commercial Code_20251111.pdf


# Gemini Test Prompt

In [15]:
def ask_legal_gemini(client, question, file_search_store_name, with_grounding=True, gemini_model="gemini-3-flash-preview"):
  response = client.models.generate_content(
            model=gemini_model,

            contents=f"""คุณเป็นผู้เชี่ยวชาญทางกฎหมายของธนาคารเอกชนแห่งหนึ่ง และมีหน้าที่ตอบคำถามดังต่อไปนี้อย่างครอบคลุม
                      โดยการตอบต้องแบ่งเป็น 3 ข้อ ดังนี้:
                      1. มาตรากฎหมายที่เกี่ยวข้อง รายการกฎหมายทั้งหมดที่จำเป็นต่อการตอบคำถาม ซึ่งต้องมาจากข้อมูลใน File Store(RAG) เท่านั้น
                      หากไม่มีมาตราใดใช้ตอบคำถามได้ ต้องแจ้งว่าข้อมูลไม่เพียงพอ และห้ามนำข้อกฎหมายจากแหล่งอื่นมาตอบเป็นอันขาด
                      โดยคำตอบจะต้องไม่มีการตัดทอนคำ สรุป หรือตัดการเว้นวรรคคำ จะต้องเป็นไปตามที่เขียนไว้ในประมวลกฎหมายเท่านั้น
                      หากคำถามสามารถมีได้หลายสถานการณ์ และใช้มาตราในการพิจารณาไม่เหมือนกัน จำเป็นต้องตอบเป็นหลายสถานการณ์

                      2. คำวินิจฉัย วินิจฉัยคำถามโดยใช้มาตรากฎหมายจากรายการในข้อที่ 1 โดยต้องตอบให้ครอบคลุม หากต้องการข้อมูลเพิ่มเติม หรือคำถามไม่ชัดเจน สามารถมีหลายคำวินิจฉัยได้ขึ้นอยู่กับสถานการณ์
                      จำเป็นต้องตอบเป็นหลายสถานการณ์ และให้คำวินิจฉัยในแต่ละสถานการณ์ให้ถูกต้อง เช่น คำวินิจฉัยแตกต่างกันหากบุคคลเป็นบุคคลธรรมดา หรือนิติบุคคล จำเป็นต้องให้คำวินิจฉัยทั้งสองสถานการณ์

                      3.ข้อสรุปและข้อเสนอแนะ จากคำวินิจฉัยข้างต้น จงสรุปและหากสามารถให้ข้อเสนอแนะ แนวทางการปฏิบัติได้ จงสรุปและให้คำแนะนำอย่างรอบคอบ

                      คำถาม:{question}""",

            # generation_config={
            #     "temperature": 0.2,  # Low = deterministic, High = creative
            #     # "max_output_tokens": 1000,
            #     # "top_p": 0.95,
            #     # "top_k": 40,
            # },

            config=types.GenerateContentConfig(
                temperature=0.2,
                tools=[
                    types.Tool(
                        file_search=types.FileSearch(
                            file_search_store_names=[file_search_store_name]
                        )
                    )
                ]
            )
  )

  # Support your response with links to the grounding sources.
  grounding = response.candidates[0].grounding_metadata
  if not grounding:
    ground_txt = 'No grounding sources found'
  else:
    sources = {c.retrieved_context.title for c in grounding.grounding_chunks}
    ground_txt = f'Sources: {sources}'

  if with_grounding:
    return response, response.text + '\n' + ground_txt

  return response, response.text

In [16]:
question="""ผู้จำนองที่เป็นผู้ถือหุ้นผู้กู้ เข้าทำสัญญาค้ำประกันเพื่อประกันหนี้ผู้กู้ ได้หรือไม่ อย่างไร"""

response, response_text = ask_legal_gemini(client, question, FILE_STORE_NAME)
print(response_text)

ในฐานะผู้เชี่ยวชาญทางกฎหมายของธนาคาร ขอสรุปและวินิจฉัยประเด็นข้อสอบถามเกี่ยวกับกรณี "ผู้จำนองที่เป็นผู้ถือหุ้นของผู้กู้ เข้าทำสัญญาค้ำประกันเพื่อประกันหนี้ของผู้กู้ได้หรือไม่" โดยอ้างอิงตามบทบัญญัติแห่งประมวลกฎหมายแพ่งและพาณิชย์ ดังนี้

### 1. มาตรากฎหมายที่เกี่ยวข้อง
จากการตรวจสอบข้อมูลใน File Store (RAG) มาตรากฎหมายที่จำเป็นในการวินิจฉัยมีดังนี้:

*   **มาตรา 709:** "บุคคลคนหนึ่งจะจำนองทรัพย์สินของตนไว้เพื่อประกันหนี้อันบุคคลอื่นจะต้องชำระ ก็ให้ทำได้" (การจำนองทรัพย์สินของตนเพื่อประกันหนี้บุคคลอื่น หรือ "ผู้จำนองลำดับสาม")
*   **มาตรา 727/1 วรรคหนึ่ง:** "ไม่ว่ากรณีจะเป็นประการใด ผู้จำนองซึ่งจำนองทรัพย์สินของตนไว้เพื่อประกันหนี้อันบุคคลอื่นจะต้องชำระ ไม่ต้องรับผิดในหนี้นั้นเกินราคาทรัพย์สินที่จำนองในเวลาที่บังคับจำนองหรือเอาทรัพย์จำนองหลุด"
*   **มาตรา 727/1 วรรคสอง:** "ข้อตกลงใดอันมีผลให้ผู้จำนองรับผิดเกินที่บัญญัติไว้ในวรรคหนึ่ง หรือให้ผู้จำนองรับผิดอย่างผู้ค้ำประกัน ข้อตกลงนั้นเป็นโมฆะ... ทั้งนี้ เว้นแต่เป็นกรณีที่นิติบุคคลเป็นลูกหนี้และบุคคลผู้มีอำนาจในการจัดการตามกฎหมายหรือบุคคลท

# Import Test Cases From File

In [17]:
test_df = pd.read_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย (ปรับเพิ่มสำหรับใช้งาน AI)_Internal Use_Extract.xlsx",
                        sheet_name="Legal Analysis to BU",
                        skiprows=[0,1,3,4])

test_df.columns = [x.strip() for x in test_df.columns]
test_df = test_df.dropna(subset=["ข้อเท็จจริง/คำถาม"])
test_df.head(3)

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,วันที่ออกความเห็น,Project/Client (กรณีบุคคลธรรมดา ให้ระบุชื่อสกุลบางส่วน เช่น สมxxx โชคxxx),BU,ประเด็นทางกฎหมาย / Key words,ข้อเท็จจริง/คำถาม,สรุปความเห็นทางกฎหมาย,รายละเอียดความเห็นทางกฎหมายแบบที่ส่งให้ BU,กฎหมายที่เกี่ยวข้อง,ฎีกาที่เกี่ยวข้อง,ผู้จัดทำความเห็น
0,17-Mar-23,NaN,CL-RE,ผู้ให้หลักประกันบัญชีเงินฝากซึ่งจด BSA ไว้เสีย...,ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเส...,ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาค...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1599 และ 160...,NaN,NaN
1,2023-01-26 00:00:00,NaN,CL-RE,สิทธิตาม BSA ในเงินฝากกับบุริมสิทธิเจ้าหนี้ค่า...,ธนาคารต้องส่งมอบเงินฝากซึ่งจด BSA ไว้ ให้แก่เจ...,กรณีที่มีบุริมสิทธิแย้งกับสิทธิตามสัญญาหลักประ...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 251, 253(3),...",NaN,NaN
2,2023-02-09 00:00:00,NaN,CL-RE,ข้อยกเว้นที่ผู้จำนองเข้าทำสัญญาค้ำประกันได้,ผู้จำนองที่เป็นผู้ถือหุ้นผู้กู้ เข้าทำสัญญาค้ำ...,ข้อตกลงใดอันมีผลให้ผู้จำนองรับรับผิดอย่างผู้ค้...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 727/1),NaN,NaN


In [18]:
test_df['AI_RESPONSE'] = test_df["ข้อเท็จจริง/คำถาม"].apply(lambda x: ask_legal_gemini(client, x, FILE_STORE_NAME)[1])

test_df.head(5)

KeyboardInterrupt: 

In [ ]:
# test_df.to_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย AI_RESPONSES_2FILES.xlsx")

In [ ]:
resp = pd.read_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย AI_RESPONSES_2FILES.xlsx")

resp.head(3)

,Unnamed: 0,วันที่ออกความเห็น,Project/Client (กรณีบุคคลธรรมดา ให้ระบุชื่อสกุลบางส่วน เช่น สมxxx โชคxxx),BU,ประเด็นทางกฎหมาย / Key words,ข้อเท็จจริง/คำถาม,สรุปความเห็นทางกฎหมาย,รายละเอียดความเห็นทางกฎหมายแบบที่ส่งให้ BU,กฎหมายที่เกี่ยวข้อง,ฎีกาที่เกี่ยวข้อง,ผู้จัดทำความเห็น,AI_RESPONSE
0,1,2023-04-30 00:00:00,ABCD,CBG,โปรดระบุประเด็นทางกฎหมายของความเห็น เช่น ผู้ให...,โปรดระบุคำถาม/ข้อเท็จจริงที่ได้รับ\nเช่น ผู้ให...,โปรดปรับ/เลือกใช้คำให้อยู่ในรูปแบบธงคำตอบ ที่ส...,โปรดระบุรายละเอียดความเห็นทางกฎหมายแบบที่ส่งให...,โปรดระบุชื่อกฎหมาย (มาตราที่เกี่ยวข้อง),โปรดระบุเลขคำพิพากษาฎีกา (หากมี),ชื่อจริง\n,"คำถาม/ข้อเท็จจริงที่ได้รับคือ: ""ผู้ให้หลักประก..."
1,2,17-Mar-23,NaN,CL-RE,ผู้ให้หลักประกันบัญชีเงินฝากซึ่งจด BSA ไว้เสีย...,ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเส...,ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาค...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1599 และ 160...,NaN,NaN,ในกรณีที่ผู้ให้หลักประกันนำบัญชีเงินฝากมาจดทะเ...
2,3,2023-01-26 00:00:00,NaN,CL-RE,สิทธิตาม BSA ในเงินฝากกับบุริมสิทธิเจ้าหนี้ค่า...,ธนาคารต้องส่งมอบเงินฝากซึ่งจด BSA ไว้ ให้แก่เจ...,กรณีที่มีบุริมสิทธิแย้งกับสิทธิตามสัญญาหลักประ...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 251, 253(3),...",NaN,NaN,เพื่อตอบคำถามของคุณว่าธนาคารต้องส่งมอบเงินฝากซ...


In [ ]:
summary_content = ""
for i, r in resp.iterrows():
  if i == 0: continue #skip example
  summary_content += "Q: " + r['ข้อเท็จจริง/คำถาม'] + "\n\n"
  summary_content += "A: " + r['สรุปความเห็นทางกฎหมาย'] + "\n\n"
  summary_content += "Gemini: " + r['AI_RESPONSE']  + "\n\n\n"

print(summary_content)

Q: ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเสียชีวิต ธนาคารยังมีบุริมสิทธิในบัญชีเงินฝากหรือไม่ และสามารถดำเนินการอย่างไรได้บ้าง
 


A: ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาคารที่มีต่อหลักประกัน เนื่องจากสิทธิในเงินฝากเป็นกองมรดกของผู้ให้หลักประกันและตกทอดไปถึงทายาทของผู้ให้หลักประกันตาม ป.พ.พ. มาตรา 1599 และ 1600 และยังเป็นหลักประกันการชำระหนี้ให้ธนาคารอยู่เนื่องจาก มาตรา 80 พ.ร.บ. หลักประกันทางธุรกิจ พ.ศ. 2558 ไม่ได้กำหนดให้ความตายของผู้ให้หลักประกันเป็นเหตุให้สัญญาหลักประกันทางธุรกิจระงับสิ้นไป

Gemini: ในกรณีที่ผู้ให้หลักประกันนำบัญชีเงินฝากมาจดทะเบียนหลักประกันทางธุรกิจ (BSA) และต่อมาผู้ให้หลักประกันเสียชีวิต ธนาคารในฐานะผู้รับหลักประกันยังมีบุริมสิทธิในบัญชีเงินฝากนั้น และสามารถดำเนินการเพื่อบังคับหลักประกันได้.

**บุริมสิทธิของธนาคาร:**
ตามพระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558 (พ.ร.บ. หลักประกันทางธุรกิจฯ) มาตรา 55 วรรคสอง บัญญัติว่า สิทธิของผู้รับหลักประกันในการบังคับหลักประกันไม่ระงับไปเพราะเหตุที่ผู้ให้หลักประกันถึงแก่ความตาย การที่ผู้ให้หลักประกันเสียชีวิตจึ

In [ ]:
with open("/content/drive/MyDrive/KKP/AI_RESPONSE_LEGAL_02.txt", 'w', encoding='utf-8') as f:
  f.write(summary_content)

# Evaluation

In [25]:
from braintrust import Eval
from autoevals import LLMClassifier, AnswerSimilarity, EmbeddingSimilarity
import os

# os.environ["BRAINTRUST_API_KEY"] = userdata.get('braintrust_api_key')

In [21]:
legal_judgement_scorer = LLMClassifier(
    name="๋2. Judgement",
    model="gemini-3-pro-preview",
    prompt_template="""คุณคือผู้เชี่ยวชาญทางกฎหมาย จงให้คะแนนความถูกต้องของข้อสอง(คำวินิจฉัย)ของคำตอบ(OUTPUT) เที่ยบกับ
    เฉลย(EXPECTED) ว่ามีความใกล้เคียงและถูกต้องแค่ไหน

    EXPECTED: {{expected}}
    OUTPUT: {{output}}""",
    choice_scores={
        "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม": 1.0,
        "ไม่ถูกต้องตามเฉลย": 0.0,
        "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [22]:
legal_suggestion_scorer = LLMClassifier(
    name="3. Conclusion & Suggestion",
    model="gemini-3-pro-preview",
    prompt_template="""คุณคือผู้เชี่ยวชาญทางกฎหมาย จงให้คะแนนความถูกต้องของข้อสาม(ข้อสรุปและข้อเสนอแนะ)ของคำตอบ(OUTPUT) เที่ยบกับ
    เฉลย(EXPECTED) ว่ามีความใกล้เคียงและถูกต้องแค่ไหน

    EXPECTED: {{expected}}
    OUTPUT: {{output}}""",
    choice_scores={
        "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม ให้ข้อเสนอแนะที่เป็นไปได้จริง": 1.0,
        "ไม่ถูกต้องตามเฉลย": 0.0,
        "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [28]:
gemini_distance = LLMClassifier(
    name="Gemini Embedding Similarity",
    model="gemini-3-pro-preview",
    prompt_template="""You are a linguistic evaluator. Rate the distance between these two sentences.
    Focus ONLY on factual entities and core predicates. Ignore tone and style.

    Sentence A: {{expected}}
    Sentence B: {{output}}""",
    choice_scores={
        "Identical Meaning": 1.0,
        "Minor Deviations": 0.8,
        "Major Deviations": 0.4,
        "Unrelated": 0.0
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [29]:
gemini_sim = LLMClassifier(
    name="Gemini Answer Similarity",
    model="gemini-3-pro-preview",
    prompt_template="""Compare the Generated Output with the Expected Answer.
    Determine if they convey the same meaning, even if the wording is different.

    Expected Answer: {{expected}}
    Generated Output: {{output}}

    Provide a similarity score based on semantic equivalence.""",
    choice_scores={
        "Equivalent": 1.0,
        "Mostly Similar": 0.7,
        "Partially Similar": 0.3,
        "Different": 0.0
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [30]:
gemini_fact = LLMClassifier(
    name="Gemini Factuality",
    model="gemini-3-pro-preview",
    prompt_template="""
You are comparing a submitted answer to an expert answer on a given question. Here is the data:
[BEGIN DATA]
************
[Question]: {{input}}
************
[Expert]: {{expected}}
************
[Submission]: {{output}}
************
[END DATA]

Compare the factual content of the submitted answer with the expert answer. Ignore any differences in style, grammar, or punctuation.
The submitted answer may either be a subset or superset of the expert answer, or it may conflict with it. Determine which case applies. Answer the question by selecting one of the following options:
(A) The submitted answer is a subset of the expert answer and is fully consistent with it.
(B) The submitted answer is a superset of the expert answer and is fully consistent with it.
(C) The submitted answer contains all the same details as the expert answer.
(D) There is a disagreement between the submitted answer and the expert answer.
(E) The answers differ, but these differences don't matter from the perspective of factuality.""",
    choice_scores={
        "A": 0.4,
        "B": 0.6,
        "C": 1,
        "D": 0,
        "E": 1
    },
    use_cot=True,
    description="Test whether an output is factual, compared to an original (`expected`) value.",
    extra_body={
        "tool_choice": "none" # Forces Gemini to use the tool call
    },
    max_tokens=None
)



In [23]:
# Prepare data for evaluation
eval_data = []
for i, r in test_df.iterrows():
    eval_data.append({
        'input': r['ข้อเท็จจริง/คำถาม'],
        # 'output': r['AI_RESPONSE'],
        'expected': r['สรุปความเห็นทางกฎหมาย']
    })

# eval_data

In [31]:
Eval(
    "KKP Legal Gemini RAG Evaluation",
    data=eval_data,
    experiment_name="Gemini-3.0 Pro V05 L&C",
    # experiment_id="63eec50f-04b2-4ab5-8060-730758dfc8e3",
    task=lambda input: ask_legal_gemini(client, input, FILE_STORE_NAME, gemini_model="gemini-3-pro-preview")[1],
    scores=[gemini_distance,
            gemini_sim,
            gemini_fact,
            legal_judgement_scorer,
            legal_suggestion_scorer]
)

Experiment Gemini-3.0 Pro V05 L&C is running at https://www.braintrust.dev/app/KKP/p/KKP%20Legal%20Gemini%20RAG%20Evaluation/experiments/Gemini-3.0%20Pro%20V05%20L%26C


<Task pending name='Task-25' coro=<_EvalCommon.<locals>.run_to_completion() running at /usr/local/lib/python3.12/dist-packages/braintrust/framework.py:759>>

In [ ]:
op = """จากข้อมูลที่ค้นพบ ไม่ได้ระบุโดยตรงเกี่ยวกับบุริมสิทธิของธนาคารในบัญชีเงินฝากที่จด BSA หลังจากผู้ให้หลักประกันเสียชีวิต และขั้นตอนการดำเนินการเฉพาะในกรณีดังกล่าว อย่างไรก็ตาม เอกสารประกอบด้วยข้อมูลทั่วไปเกี่ยวกับการให้บริการรับฝากเงินของธนาคารพาณิชย์และบริษัทเงินทุนภายใต้การกำกับดูแลของธนาคารแห่งประเทศไทย.

ตามประกาศธนาคารแห่งประเทศไทย ธนาคารพาณิชย์และบริษัทเงินทุนสามารถให้บริการรับฝากเงินที่มีเงื่อนไขการเบิกถอนเงินจากบัญชีตามคำสั่งลูกค้าได้ โดยต้องปฏิบัติตามหลักเกณฑ์ที่กำหนด. ในการเปิดบัญชีที่ให้บริการดังกล่าว จะต้องไม่ใช้ชื่อบัญชีดูแลผลประโยชน์ตามพระราชบัญญัติการดูแลผลประโยชน์ของคู่สัญญา พ.ศ. 2551 และต้องระบุให้ชัดเจนว่าเงินในบัญชีดังกล่าวจะได้รับการคุ้มครองเช่นเดียวกับเงินฝากทั่วไปตามที่สถาบันคุ้มครองเงินฝากกำหนด ไม่ใช่การคุ้มครองภายใต้พระราชบัญญัติการดูแลผลประโยชน์ของคู่สัญญาฯ.

นอกจากนี้ ในสัญญาประกอบการเปิดบัญชีจะต้องมีการระบุรายละเอียดต่างๆ เช่น ชื่อและที่อยู่ของลูกค้า ขอบเขตความรับผิดชอบและบทบาทหน้าที่ของธนาคาร หลักเกณฑ์การเปิดบัญชี การฝากหรือเบิกถอนเงิน การชำระดอกเบี้ย การปิดบัญชี รายละเอียดค่าธรรมเนียม และวันเริ่มต้นและสิ้นสุดของบัญชี. ธนาคารยังมีหน้าที่ต้องกำหนดนโยบายและขั้นตอนการปฏิบัติงานเป็นลายลักษณ์อักษร เช่น ระเบียบการเปิดบัญชี การรับฝากหรือถอนเงิน การจ่ายดอกเบี้ย การคิดค่าธรรมเนียม และระบบการควบคุมภายใน. ธนาคารต้องจัดให้มีรายละเอียดเกี่ยวกับสิทธิของลูกค้าต่อเงินฝากและแจ้งให้ลูกค้าทราบและเข้าใจถึงสิทธิดังกล่าวอย่างชัดเจน.

สำหรับกรณีการเสียชีวิตของผู้ให้หลักประกันที่มีการนำบัญชีเงินฝากมาจด BSA นั้น โดยทั่วไปแล้ว สิทธิและหน้าที่ตามสัญญาหลักประกันมักจะยังคงอยู่และตกทอดไปยังกองมรดกของผู้เสียชีวิต แต่รายละเอียดเกี่ยวกับบุริมสิทธิของธนาคารและขั้นตอนการดำเนินการทางกฎหมาย เช่น การแจ้งทายาท การทวงหนี้ หรือการบังคับหลักประกัน จะต้องพิจารณาตามข้อตกลงในสัญญา BSA และกฎหมายแพ่งและพาณิชย์เกี่ยวกับมรดกและหลักประกัน ซึ่งข้อมูลเหล่านี้ไม่ปรากฏในเอกสารที่ค้นพบ.
Sources: {'20240702_BOT_Policy.pdf'}"""

exp =     """ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาคารที่มีต่อหลักประกัน เนื่องจากสิทธิในเงินฝากเป็นกองมรดกของผู้ให้หลักประกันและตกทอดไปถึงทายาทของผู้ให้หลักประกันตาม ป.พ.พ. มาตรา 1599 และ 1600 และยังเป็นหลักประกันการชำระหนี้ให้ธนาคารอยู่เนื่องจาก มาตรา 80 พ.ร.บ. หลักประกันทางธุรกิจ พ.ศ. 2558 ไม่ได้กำหนดให้ความตายของผู้ให้หลักประกันเป็นเหตุให้สัญญาหลักประกันทางธุรกิจระงับสิ้นไป"""
inp = """ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเสียชีวิต ธนาคารยังมีบุริมสิทธิในบัญชีเงินฝากหรือไม่ และสามารถดำเนินการอย่างไรได้บ้าง"""

for scorer in [gemini_fact, gemini_sim, gemini_distance]:
  result_score = scorer(op, exp, input=inp)
  print(result_score.name)
  print(result_score.score)
  print(result_score.metadata['rationale'])
  print('\n\n')

Gemini Factuality
0.4
1. **Analyze the Expert Answer**: The expert answer states that the death of the security provider does not affect the bank's rights because the deposit becomes part of the estate (citing CCC 1599, 1600) and the Business Security Act (BSA) Section 80 does not list death as a reason for contract termination.

2. **Analyze the Submitted Answer**:
    - It starts by stating that the specific information about preferential rights after death was not found in its source documents.
    - It provides extensive but irrelevant information about Bank of Thailand (BOT) regulations regarding deposit accounts.
    - It concludes by stating that "generally" rights and duties under a security contract remain and pass to the estate, but specific details depend on the BSA contract and law, which were not in its sources.

3. **Compare Factual Content**:
    - **Consistency**: Both answers agree on the general principle that the rights over the security (the deposit account) continu